# Step 8 – Robustness checks

This notebook reproduces the robustness checks of report **section 4.8** and the results that depend on them in **sections 5.2 and 5.4**:

| Part | What is tested | Report |
|---|---|---|
| 1 | Sensitivity of the pressure levels to the assumed **effective width** | §5.2 |
| 2 | Whether the **busiest** location is also the most **heavily loaded** one | §5.2 |
| 3 | What happened **after** a sudden-change flag | Table 5.2 |
| 4 | Whether sudden increases give **early warning** of high pressure, compared with chance | Table 5.3 |
| 5 | Sensitivity of the number of flags to the **detection parameters** | Table 5.4 |

All calculations reuse the functions of the dashboard (`Dashboard/sail_analysis.py`), so the numbers are based on exactly the same cleaned data, indicators, pressure levels and sudden-change method as the dashboard and the report. At the end, every reproduced value is compared with the value in the report.

**Run order:** `Data/SAIL2025_cleaned.ipynb` → `step4/SAIL2025_indicators_step4.ipynb` → this notebook.

## 0. Preparation

In [6]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make the dashboard modules importable (this notebook lives in step8/, the code in Dashboard/)
DASHBOARD_DIR = Path.cwd().parent / "Dashboard"
sys.path.insert(0, str(DASHBOARD_DIR))

import config as cfg
import sail_analysis as sa

SEED = 2025          # fixed seed: the random tests give the same result every run
N_DRAWS = 2000       # number of random draws / shifts, as in report section 4.8
ATTENTION = cfg.LEVEL_FROM["attention"]   # q >= 33 ped/min/m (Table 2.1)

In [7]:
data = sa.load()                    # cleaned counts + locations + indicators + levels + sudden changes
base = data.base.sort_values(["sensor", "timestamp"]).reset_index(drop=True)
events = data.events.copy()

# Sensor groups (report section 3.1): ferry landings are the GVCV sensors
base["ferry"] = base["sensor"].str.startswith("GVCV")
events["ferry"] = events["sensor"].str.startswith("GVCV")

valid = base[base["status"] == "ok"].copy()   # missing and suspect intervals are excluded (section 4.3)

print(f"{base['sensor'].nunique()} sensors, {len(base):,} sensor x interval rows, {len(valid):,} valid")
print(f"{len(events)} sudden changes: {(events['type'] == 'increase').sum()} increases, "
      f"{(events['type'] == 'decrease').sum()} decreases")

35 sensors, 84,000 sensor x interval rows, 83,121 valid
459 sudden changes: 319 increases, 140 decreases


## 1. Sensitivity to the effective width (report §5.2)

The flow per metre $q = N / (3 \cdot w)$ depends directly on the effective width $w$, which is an assumption in the location file. We therefore recalculate the results with:

1. all widths **10% smaller** and **10% larger**;
2. the **gross width of 3.5 m** at the ferry landings instead of the effective 3.1 m;
3. **2,000 random draws** in which the width of *each* sensor varies **independently** within ±10%.

For 1 and 2 we count the intervals above the normal level ($q \ge 33$) and the maximum $q$. For 3 we check how stable the **top 5** of locations by peak flow per metre is.

In [8]:
def flow_per_metre(df, width):
    # q in ped/min/m for a given width (Series aligned with df, or a scalar factor applied to the widths)
    return df["N_total"] / (cfg.INTERVAL_MIN * width)

rows = []
for name, width in [("Widths -10%", valid["eff_width_m"] * 0.9),
                    ("Base case",   valid["eff_width_m"]),
                    ("Widths +10%", valid["eff_width_m"] * 1.1),
                    ("Ferry landings 3.5 m", valid["eff_width_m"].where(~valid["ferry"], 3.5))]:
    q = flow_per_metre(valid, width)
    rows.append({"scenario": name,
                 "intervals above normal (q >= 33)": int((q >= ATTENTION).sum()),
                 "max q (ped/min/m)": round(q.max(), 1),
                 "intervals at crisis (q >= 82)": int((q >= cfg.LEVEL_FROM["crisis"]).sum())})
width_table = pd.DataFrame(rows)
width_table

,scenario,intervals above normal (q >= 33),max q (ped/min/m),intervals at crisis (q >= 82)
0,Widths -10%,664,65.4,0
1,Base case,330,58.8,0
2,Widths +10%,180,53.5,0
3,Ferry landings 3.5 m,186,52.1,0


**Interpretation:** a 10% change in width changes the number of intervals above normal by a factor of about two (180–664), but the maximum stays far below the crisis level. The *magnitude* is sensitive to the width; whether a crisis level was reached is not.

In [9]:
# Peak flow per metre per sensor in the base case, and its top 5
peak_N = valid.groupby("sensor")["N_total"].max()
width_per_sensor = valid.groupby("sensor")["eff_width_m"].first()
peak_q_base = peak_N / (cfg.INTERVAL_MIN * width_per_sensor)
top5_base = set(peak_q_base.nlargest(5).index)
leader_base = peak_q_base.idxmax()

rng = np.random.default_rng(SEED)
leader_kept, overlap = 0, []
for _ in range(N_DRAWS):
    factor = pd.Series(rng.uniform(0.9, 1.1, len(width_per_sensor)), index=width_per_sensor.index)
    peak_q = peak_N / (cfg.INTERVAL_MIN * width_per_sensor * factor)   # q scales with 1/w, so the peak count gives the peak q
    leader_kept += peak_q.idxmax() == leader_base
    overlap.append(len(top5_base & set(peak_q.nlargest(5).index)))

print(f"Base-case leader: {data.sensors.set_index('sensor').loc[leader_base, 'label']}")
print(f"Leader stays first in {leader_kept / N_DRAWS:.0%} of {N_DRAWS} draws")
print(f"On average {np.mean(overlap):.1f} of the top 5 stay the same")

Base-case leader: NDSMkade · GVCV-11
Leader stays first in 92% of 2000 draws
On average 4.7 of the top 5 stay the same


**Interpretation:** even when every width varies independently, NDSM-kade stays first in about nine out of ten draws and almost the whole top 5 stays the same. The *ranking* of locations is robust; counts of intervals above a threshold should be read as indications.

## 2. Busiest versus most heavily loaded location (report §5.2)

Does ranking by absolute count give the same locations as ranking by flow per metre? We compare the peak count and the peak $q$ per sensor with the **Spearman rank correlation** (1 = identical order, 0 = no relation) and count how many sensors appear in both top fives.

In [10]:
peaks = valid.groupby("sensor").agg(peak_N=("N_total", "max"), peak_q=("q_specific", "max"))
rho = peaks["peak_N"].rank().corr(peaks["peak_q"].rank())   # Spearman = correlation of the ranks (no scipy needed)
both = set(peaks["peak_N"].nlargest(5).index) & set(peaks["peak_q"].nlargest(5).index)
print(f"Spearman rank correlation peak count vs peak q: {rho:.2f}")
print(f"Sensors in both top fives: {len(both)} {sorted(both)}")

Spearman rank correlation peak count vs peak q: 0.50
Sensors in both top fives: 1 ['GVCV-11']


## 3. What followed a sudden-change flag (Table 5.2)

For every flag we look at the period **after** the flagged interval:

- **Lasted 15 min:** the mean count of the next 15 minutes (5 intervals) is closer to the **new** level (count at the flag) than to the **old** level (the 15-minute reference before the change).
- **Reached Attention within 30 min:** $q \ge 33$ in at least one of the next 10 intervals.

In [11]:
by_sensor = {s: g.set_index("timestamp") for s, g in base.groupby("sensor")}

def after_flag(row, minutes, column):
    # Values of `column` in the `minutes` after the flagged interval (flag itself excluded)
    g = by_sensor[row.sensor]
    window = (g.index > row.flagged_at) & (g.index <= row.flagged_at + pd.Timedelta(minutes=minutes))
    return g.loc[window, column]

follow = []
for row in events.itertuples():
    next_15 = after_flag(row, 15, "N_total").mean()
    follow.append({
        "lasted_15": abs(next_15 - row.N_at_flag) < abs(next_15 - row.N_before),
        "attention_30": (after_flag(row, 30, "q_specific") >= ATTENTION).any(),
    })
events = events.join(pd.DataFrame(follow))

table_5_2 = (events.assign(sensors=np.where(events["ferry"], "Ferry landings", "Other sensors"))
                   .groupby(["sensors", "type"])
                   .agg(flags=("type", "size"),
                        lasted_15_min=("lasted_15", "mean"),
                        reached_attention_30_min=("attention_30", "mean")))
table_5_2[["lasted_15_min", "reached_attention_30_min"]] = (table_5_2[["lasted_15_min", "reached_attention_30_min"]] * 100).round(0)
table_5_2   # percentages; the report leaves out the 2 decreases at ferry landings

flags  lasted_15_min  reached_attention_30_min
sensors        type                                                    
Ferry landings decrease      2           50.0                       0.0
               increase    186           10.0                      16.0
Other sensors  decrease    138           20.0                       1.0
               increase    133           32.0                       0.0

## 4. Do sudden increases give early warning? (Table 5.3)

### 4.1 Episodes of high pressure
An **episode** is a period in which a sensor is at or above a pressure threshold. Intervals above the threshold belong to the same episode unless the gap between them is **more than 30 minutes**.

In [12]:
def find_episodes(df, threshold, max_gap_min=30):
    # One row per episode: sensor, start, end and number of intervals at or above `threshold`
    episodes = []
    for sensor, g in df[df["q_specific"] >= threshold].groupby("sensor"):
        new_episode = g["timestamp"].diff() > pd.Timedelta(minutes=max_gap_min)
        for _, e in g.groupby(new_episode.cumsum()):
            episodes.append({"sensor": sensor, "start": e["timestamp"].min(),
                             "end": e["timestamp"].max(), "n_intervals": len(e)})
    out = pd.DataFrame(episodes)
    out["ferry"] = out["sensor"].str.startswith("GVCV")
    return out

episodes_33 = find_episodes(valid, ATTENTION)
episodes_25 = find_episodes(valid, 25)

print("Episodes at q >= 33:", episodes_33.groupby("ferry").size().rename({True: "ferry", False: "other"}).to_dict())
print(f"Ferry episodes that last a single interval: {(episodes_33.loc[episodes_33['ferry'], 'n_intervals'] == 1).mean():.0%}")
print("Episodes at q >= 25 at other sensors:", (~episodes_25["ferry"]).sum())

Episodes at q >= 33: {'other': 13, 'ferry': 109}
Ferry episodes that last a single interval: 60%
Episodes at q >= 25 at other sensors: 63


### 4.2 Observed number of episodes preceded by a sudden increase
An episode counts as "announced" if the same sensor had a sudden **increase** flagged in the 15 (or 30) minutes **before** the start of the episode.

In [13]:
# Work in minutes since the start of the event: fast and easy to shift
T0 = base["timestamp"].min()
to_min = lambda ts: ((ts - T0).dt.total_seconds() / 60).to_numpy()

inc = events[events["type"] == "increase"]
flag_sensor, flag_min = inc["sensor"].to_numpy(), to_min(inc["flagged_at"])

def n_announced(flag_sensor, flag_min, ep_sensor, ep_start, window):
    # Number of episodes with at least one flag in [start - window, start)
    count = 0
    for s, start in zip(ep_sensor, ep_start):
        lead = start - flag_min[flag_sensor == s]
        count += np.any((lead > 0) & (lead <= window))
    return count

tests = {   # name: (episodes, window in minutes)
    "Ferry landings, q >= 33, 15 min": (episodes_33[episodes_33["ferry"]], 15),
    "Ferry landings, q >= 33, 30 min": (episodes_33[episodes_33["ferry"]], 30),
    "Other sensors, q >= 33, 15 min":  (episodes_33[~episodes_33["ferry"]], 15),
    "Other sensors, q >= 25, 15 min":  (episodes_25[~episodes_25["ferry"]], 15),
    "Other sensors, q >= 25, 30 min":  (episodes_25[~episodes_25["ferry"]], 30),
}
tests = {k: (e["sensor"].to_numpy(), to_min(e["start"]), w, len(e)) for k, (e, w) in tests.items()}
observed = {k: n_announced(flag_sensor, flag_min, s, st, w) for k, (s, st, w, _) in tests.items()}
observed

{'Ferry landings, q >= 33, 15 min': np.int64(6),
 'Ferry landings, q >= 33, 30 min': np.int64(10),
 'Other sensors, q >= 33, 15 min': np.int64(0),
 'Other sensors, q >= 25, 15 min': np.int64(2),
 'Other sensors, q >= 25, 30 min': np.int64(3)}

### 4.3 Comparison with chance
Some matches arise by coincidence, especially at sensors with many flags. To estimate how many, we **shift all flags of a sensor** by a random number of **days (1–4)** plus at most **±2 hours**, wrapping around the 5-day period. This keeps the number of flags per sensor and their daily rhythm, but breaks their link with the episodes. Repeating this 2,000 times gives the distribution of matches **by chance**.

The **p-value** is the share of random shifts that give at least as many matches as observed. A high p-value means the observed number is what you would expect by coincidence.

In [14]:
PERIOD = 5 * 24 * 60                       # 5 event days in minutes
sensors_with_flags = np.unique(flag_sensor)
rng = np.random.default_rng(SEED)

chance = {k: np.empty(N_DRAWS) for k in tests}
for i in range(N_DRAWS):
    shift = np.zeros(len(flag_min))
    for s in sensors_with_flags:            # one shift per sensor
        shift[flag_sensor == s] = rng.integers(1, 5) * 1440 + rng.integers(-40, 41) * cfg.INTERVAL_MIN
    shifted = (flag_min + shift) % PERIOD
    for k, (s, st, w, _) in tests.items():
        chance[k][i] = n_announced(flag_sensor, shifted, s, st, w)

table_5_3 = pd.DataFrame([{
    "episodes": k,
    "preceded by a flag": f"{observed[k]} of {tests[k][3]}",
    "expected by chance": round(chance[k].mean(), 1),
    "95% range": f"{np.percentile(chance[k], 2.5):.0f}–{np.percentile(chance[k], 97.5):.0f}",
    "p": round((chance[k] >= observed[k]).mean(), 2),
} for k in tests])
table_5_3

,episodes,preceded by a flag,expected by chance,95% range,p
0,"Ferry landings, q >= 33, 15 min",6 of 109,6.3,2–11,0.62
1,"Ferry landings, q >= 33, 30 min",10 of 109,12.3,7–18,0.83
2,"Other sensors, q >= 33, 15 min",0 of 13,0.3,0–2,1.00
3,"Other sensors, q >= 25, 15 min",2 of 63,1.7,0–5,0.54
4,"Other sensors, q >= 25, 30 min",3 of 63,3.5,0–7,0.70


**Interpretation:** the observed numbers are at or below what chance alone gives (high p-values). Sudden increases therefore do **not** announce high pressure. Because this is a random test, the expected values and p-values can differ by a few hundredths from the report depending on the random draws; the conclusion does not change.

## 5. Sensitivity to the detection parameters (Table 5.4)

The sudden-change method has three parameters (report §4.5): the threshold **k** (number of standard deviations), the **persistence** (number of consecutive exceeding intervals) and the minimum level **N_min** (pedestrians per 3 min). We vary one at a time around the chosen setting (k = 2, persistence = 2, N_min = 50).

In [15]:
settings = [(2, 2, 50), (1.5, 2, 50), (2.5, 2, 50), (3, 2, 50),
            (2, 1, 50), (2, 3, 50), (2, 2, 30), (2, 2, 100)]
n_sensors, n_days = base["sensor"].nunique(), base["date"].nunique()

rows = []
for k, persistence, n_min in settings:
    _, ev = sa.detect_sudden_changes(data.base.copy(), k=k, persistence=persistence, min_level=n_min)
    rows.append({"k": k, "persistence": persistence, "N_min": n_min, "flags": len(ev),
                 "flags per sensor per day": round(len(ev) / n_sensors / n_days, 1),
                 "share at ferry landings (%)": round(ev["sensor"].str.startswith("GVCV").mean() * 100)})
table_5_4 = pd.DataFrame(rows)
table_5_4   # first row = chosen setting

,k,persistence,N_min,flags,flags per sensor per day,share at ferry landings (%)
0,2.0,2,50,459,2.6,41
1,1.5,2,50,1052,6.0,27
2,2.5,2,50,232,1.3,57
3,3.0,2,50,127,0.7,65
4,2.0,1,50,5662,32.4,29
5,2.0,3,50,17,0.1,47
6,2.0,2,30,589,3.4,40
7,2.0,2,100,261,1.5,33


## 6. Check against the report

In [16]:
t52 = table_5_2.reset_index()
get = lambda sensors, typ, col: t52.loc[(t52.sensors == sensors) & (t52.type == typ), col].iloc[0]

check = pd.DataFrame([
    ("§5.2 Intervals above normal, widths -10% / +10%", "664 / 180",
     f"{width_table.iloc[0, 1]} / {width_table.iloc[2, 1]}"),
    ("§5.2 Max q, widths +10% / -10%", "53.5 / 65.4",
     f"{width_table.iloc[2, 2]} / {width_table.iloc[0, 2]}"),
    ("§5.2 Peak q with ferry width 3.5 m", "52.1", f"{width_table.iloc[3, 2]}"),
    ("§5.2 NDSM-kade first in random draws", "92%", f"{leader_kept / N_DRAWS:.0%}"),
    ("§5.2 Top 5 kept on average", "4.7", f"{np.mean(overlap):.1f}"),
    ("§5.2 Rank correlation peak count vs peak q", "0.50", f"{rho:.2f}"),
    ("Table 5.2 Lasted 15 min (ferry inc / other inc / other dec)", "10% / 32% / 20%",
     f"{get('Ferry landings','increase','lasted_15_min'):.0f}% / {get('Other sensors','increase','lasted_15_min'):.0f}% / "
     f"{get('Other sensors','decrease','lasted_15_min'):.0f}%"),
    ("Table 5.3 Episodes q >= 33 (ferry / other)", "109 / 13",
     f"{episodes_33['ferry'].sum()} / {(~episodes_33['ferry']).sum()}"),
    ("Table 5.3 Ferry episodes preceded by a flag (15 min)", "6", f"{observed['Ferry landings, q >= 33, 15 min']}"),
    ("Table 5.4 Flags at the chosen setting", "459", f"{table_5_4.loc[0, 'flags']}"),
    ("Table 5.4 Flags without persistence", "5,662", f"{table_5_4.loc[4, 'flags']:,}"),
], columns=["result", "report", "reproduced"])
check["match"] = np.where(check["report"].str.replace(",", "") == check["reproduced"].str.replace(",", ""), "✅", "check")
check

,result,report,reproduced,match
0,"§5.2 Intervals above normal, widths -10% / +10%",664 / 180,664 / 180,✅
1,"§5.2 Max q, widths +10% / -10%",53.5 / 65.4,53.5 / 65.4,✅
2,§5.2 Peak q with ferry width 3.5 m,52.1,52.1,✅
3,§5.2 NDSM-kade first in random draws,92%,92%,✅
4,§5.2 Top 5 kept on average,4.7,4.7,✅
5,§5.2 Rank correlation peak count vs peak q,0.50,0.50,✅
6,Table 5.2 Lasted 15 min (ferry inc / other inc...,10% / 32% / 20%,10% / 32% / 20%,✅
7,Table 5.3 Episodes q >= 33 (ferry / other),109 / 13,109 / 13,✅
8,Table 5.3 Ferry episodes preceded by a flag (1...,6,6,✅
9,Table 5.4 Flags at the chosen setting,459,459,✅


## Conclusion

All robustness results of report section 4.8 and the values in sections 5.2 and 5.4 can be reproduced from the cleaned data with the functions of the dashboard. The main findings hold:

- **Width:** the location ranking is robust; the number of intervals above a threshold is sensitive to the assumed width and should be read as an indication.
- **Peaks:** the busiest locations are not the most heavily loaded ones (rank correlation 0.50).
- **Sudden changes:** most flags mark short fluctuations, and sudden increases do not announce high pressure better than chance. Their value lies in revealing abrupt events, not in early warning.
- **Parameters:** the number of flags depends strongly on persistence; the chosen setting is a compromise, not a calibrated optimum.